In [ ]:
import os
import random
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

In [ ]:
# ── Dataset Configuration ──────────────────────────────────────────────────
DATA_DIR = '/kaggle/input/datasets/fatihkgg/affectnet-yolo-format/YOLO_format'

# AffectNet 8-class → CalmSpace 4-class mapping
# Original IDs: 0=anger 1=contempt 2=disgust 3=fear 4=happy 5=neutral 6=sad 7=surprise
#
# FIX vs old notebook:
#   - Surprise (7) moved OUT of Calm → its own Distressed bucket
#     (open mouth + raised brows looks Distressed, not Calm)
#   - Contempt (1) moved to Overwhelmed (not Distressed)
#   - This makes the 4 classes visually much more distinct
CLASS_MAP = {
    5: 0,  # neutral   → Calm
    4: 1,  # happy     → Happy
    6: 2,  # sad       → Distressed
    3: 2,  # fear      → Distressed
    7: 2,  # surprise  → Distressed  ← moved from Calm
    0: 3,  # anger     → Overwhelmed
    1: 3,  # contempt  → Overwhelmed
    2: 3,  # disgust   → Overwhelmed
}
CLASS_NAMES = ['Calm', 'Happy', 'Distressed', 'Overwhelmed']


class AffectNetDataset(Dataset):
    def __init__(self, split='train', transform=None, max_samples=5000):
        self.transform = transform
        images_dir = os.path.join(DATA_DIR, split, 'images')
        labels_dir = os.path.join(DATA_DIR, split, 'labels')
        self.data = []
        class_counts = defaultdict(list)

        if not os.path.exists(images_dir):
            print(f'WARNING: {images_dir} does not exist')
            return

        for img_name in os.listdir(images_dir):
            if not img_name.endswith(('.jpg', '.png')):
                continue
            lbl_path = os.path.join(labels_dir, img_name.rsplit('.', 1)[0] + '.txt')
            if not os.path.exists(lbl_path):
                continue
            with open(lbl_path) as f:
                content = f.read().strip().split()
            if not content:
                continue
            orig_class = int(content[0])
            mapped = CLASS_MAP.get(orig_class)
            if mapped is not None:
                class_counts[mapped].append((os.path.join(images_dir, img_name), mapped))

        # Hard-balance: same number of samples per class
        for cls, items in class_counts.items():
            sampled = random.sample(items, min(len(items), max_samples))
            self.data.extend(sampled)
            print(f'  Class {CLASS_NAMES[cls]}: {len(sampled)} samples')

        random.shuffle(self.data)
        print(f'Total {split} samples: {len(self.data)}')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label


# Augmentation-heavy train transform; clean val transform
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print('Loading datasets...')
train_dataset = AffectNetDataset('train', train_transform, max_samples=5000)
val_dataset   = AffectNetDataset('valid', val_transform,   max_samples=1000)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
print('Data ready!')

In [ ]:
# ── Model Setup ────────────────────────────────────────────────────────────
# KEY FIX: unfreeze the last 3 feature blocks + train the full classifier.
# Old notebook only trained the final linear layer on frozen ImageNet features
# — not enough for subtle face expressions.

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

# Freeze early layers (low-level edges/textures — reusable from ImageNet)
for param in model.features[:14].parameters():
    param.requires_grad = False

# Unfreeze last 4 feature blocks (high-level semantic features — need face tuning)
for param in model.features[14:].parameters():
    param.requires_grad = True

# Replace classifier head for 4 classes with dropout for regularisation
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(model.last_channel, 4),
)

model = model.to(device)

# Two learning rates: small LR for unfrozen feature blocks, bigger LR for new head
feature_params    = [p for p in model.features[14:].parameters() if p.requires_grad]
classifier_params = list(model.classifier.parameters())

optimizer = optim.Adam([
    {'params': feature_params,    'lr': 1e-4},
    {'params': classifier_params, 'lr': 5e-4},
])

# LR scheduler: reduce on plateau if val accuracy stops improving
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)

criterion = nn.CrossEntropyLoss()
print('Model ready!')

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,}  |  Trainable: {trainable_params:,}')

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────────────
EPOCHS   = 12  # More epochs with scheduler — stops overfitting automatically
best_acc = 0.0

for epoch in range(EPOCHS):
    # ── Train ──
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # ── Validate ──
    model.eval()
    correct = 0
    total   = 0
    class_correct = [0] * 4
    class_total   = [0] * 4

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            preds = model(inputs).argmax(dim=1)
            total   += labels.size(0)
            correct += (preds == labels).sum().item()
            for i in range(4):
                mask = labels == i
                class_total[i]   += mask.sum().item()
                class_correct[i] += (preds[mask] == i).sum().item()

    val_acc = 100 * correct / total
    per_class = '  '.join(
        f"{CLASS_NAMES[i]}:{100*class_correct[i]/max(class_total[i],1):.0f}%"
        for i in range(4)
    )
    avg_loss = running_loss / len(train_loader)
    print(f'Ep {epoch+1:02d}/{EPOCHS} | Loss: {avg_loss:.3f} | Val: {val_acc:.1f}%  [{per_class}]')

    scheduler.step(val_acc)

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), '/kaggle/working/best_model.pth')
        print(f'  ✓ Saved best model  (val_acc={val_acc:.1f}%)')

print(f'\nTRAINING COMPLETE — Best val accuracy: {best_acc:.1f}%')
print('Download best_model.pth from the Output panel on the right.')